<a href="https://colab.research.google.com/github/Rakshay94/Deta-analyst-pw-skills-/blob/ML/Cryptocurrency_Liquidity_Prediction_ML_Project_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Project

# Cryptocurrency Liquidity
# Prediction for Market Stability

In [4]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy.stats import randint
import joblib # For saving the model for deployment

# --- 1. Data Collection & Initial Setup ---
print("--- Step 1: Data Collection & Initial Setup ---")
# Load the datasets
# Assuming 'coin_gecko_2022-03-16.csv' and 'coin_gecko_2022-03-17.csv' are in the same directory
try:
    df_2022_03_16 = pd.read_csv('coin_gecko_2022-03-16.csv')
    df_2022_03_17 = pd.read_csv('coin_gecko_2022-03-17.csv')
    print("CSV files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading CSV files: {e}. Please ensure they are in the correct directory.")
    exit() # Exit if files are not found

# Combine the dataframes
combined_df = pd.concat([df_2022_03_16, df_2022_03_17], ignore_index=True)

# Convert 'date' column to datetime objects for time-based operations
combined_df['date'] = pd.to_datetime(combined_df['date'])

print("\nCombined DataFrame head (before extensive preprocessing):")
print(combined_df.head().to_markdown(index=False, numalign="left", stralign="left"))
print(f"\nInitial combined DataFrame shape: {combined_df.shape}")
print("\nInitial missing values overview:")
print(combined_df.isnull().sum().to_markdown(numalign="left", stralign="left"))


# --- 2. Data Preprocessing ---
print("\n--- Step 2: Data Preprocessing ---")

# Handle missing values
# Impute missing numerical values with the median
# First, identify numerical columns that might have NaNs introduced by the initial load
numerical_cols_to_impute = ['1h', '24h', '7d', '24h_volume', 'price', 'mkt_cap']
imputer = SimpleImputer(strategy='median')
for col in numerical_cols_to_impute:
    if col in combined_df.columns:
        combined_df[col] = imputer.fit_transform(combined_df[[col]])

# Ensure no more missing values in critical columns post-imputation
print("\nMissing values after numerical imputation:")
print(combined_df[numerical_cols_to_impute].isnull().sum().to_markdown(numalign="left", stralign="left"))

# Drop columns that are not useful for the model or were just for identification
# (e.g., 's.no' if present, 'sn_id' if present)
cols_to_drop_if_present = ['s.no', 'sn_id']
for col in cols_to_drop_if_present:
    if col in combined_df.columns:
        combined_df = combined_df.drop(columns=[col])

print("\nDataFrame info after initial cleaning and imputation:")
combined_df.info()


# --- 3. Feature Engineering ---
print("\n--- Step 3: Feature Engineering ---")

# Feature 1: Price-Volume Ratio
# Add a small epsilon to avoid division by zero for 24h_volume
epsilon = 1e-6
combined_df['price_volume_ratio'] = combined_df['price'] / (combined_df['24h_volume'] + epsilon)

# Feature 2: Market Cap-Volume Ratio
combined_df['market_cap_volume_ratio'] = combined_df['mkt_cap'] / (combined_df['24h_volume'] + epsilon)

# Feature 3: Volatility (using standard deviation of price changes)
# We use '1h', '24h', '7d' for this. Ensure they are numeric.
combined_df['volatility'] = combined_df[['1h', '24h', '7d']].std(axis=1)

# Handle potential infinite values resulting from division by zero, if any, replace with NaN
combined_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Impute any new NaN values created by feature engineering (e.g., if original 24h_volume was 0)
new_engineered_cols = ['price_volume_ratio', 'market_cap_volume_ratio', 'volatility']
for col in new_engineered_cols:
    if combined_df[col].isnull().any():
        median_val = combined_df[col].median()
        combined_df[col].fillna(median_val, inplace=True)

# Feature 4: Time since epoch (for temporal insights)
# Convert datetime to Unix timestamp (seconds since epoch)
combined_df['time_since_epoch'] = combined_df['date'].astype(np.int64) // 10**9

print("\nDataFrame head with newly engineered features:")
print(combined_df[['price', '24h_volume', 'mkt_cap', '1h', '24h', '7d',
                   'price_volume_ratio', 'market_cap_volume_ratio', 'volatility', 'time_since_epoch']].head().to_markdown(index=False, numalign="left", stralign="left"))


# --- 4. Exploratory Data Analysis (EDA) - Conceptual Code ---
# (Note: For full visualizations, uncomment and run in an environment with matplotlib/seaborn)
print("\n--- Step 4: Exploratory Data Analysis (EDA) - Conceptual Snippets ---")
print("Descriptive Statistics of Numerical Features:")
# Select only numeric types after all preprocessing for describe()
print(combined_df.select_dtypes(include=np.number).describe().to_markdown(numalign="left", stralign="left"))

# Calculate and display correlation matrix for key numerical features
numerical_and_target_cols = ['price', '1h', '24h', '7d', '24h_volume', 'mkt_cap',
                             'price_volume_ratio', 'market_cap_volume_ratio', 'volatility', 'time_since_epoch']
correlation_matrix = combined_df[numerical_and_target_cols].corr()
print("\nCorrelation Matrix of Numerical Features (first few rows):")
print(correlation_matrix.head().to_markdown(numalign="left", stralign="left"))

# Conceptual placeholders for visualizations:
# import matplotlib.pyplot as plt
# import seaborn as sns
# plt.figure(figsize=(10, 6))
# sns.histplot(combined_df['24h_volume'], kde=True, bins=50)
# plt.title('Distribution of 24h Volume (Liquidity)')
# plt.xlabel('24h Volume')
# plt.ylabel('Frequency')
# plt.show()
#
# plt.figure(figsize=(12, 10))
# sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
# plt.title('Correlation Matrix of Numerical Features')
# plt.show()


# --- 5. Feature Scaling & Encoding (Final stage before model training) ---
print("\n--- Step 5: Feature Scaling & Encoding (Final Stage) ---")

# Separate numerical and categorical columns for scaling and encoding
# Exclude 'date' and '24h_volume' (target) from numerical_features for scaling
numerical_features_for_scaling = [col for col in combined_df.select_dtypes(include=np.number).columns if col not in ['24h_volume']] # 'date' is already converted to 'time_since_epoch'

# Initialize StandardScaler and apply to numerical features
scaler = StandardScaler()
combined_df[numerical_features_for_scaling] = scaler.fit_transform(combined_df[numerical_features_for_scaling])

# One-hot encode 'coin' and 'symbol' columns (if not already done in step 2 and if they exist)
# Re-apply encoding in case previous steps changed structure or new categories emerged, ensuring consistency.
# In this specific case, it's done early in preprocessing, but this is where it'd typically be after splitting for a robust pipeline.
# For this consolidated script, we ensure it's handled.
# If 'coin' and 'symbol' are still present as non-numeric, encode them.
if 'coin' in combined_df.columns and combined_df['coin'].dtype == 'object':
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoded_features = encoder.fit_transform(combined_df[['coin', 'symbol']])
    encoded_feature_names = encoder.get_feature_names_out(['coin', 'symbol'])
    encoded_df = pd.DataFrame(encoded_features, columns=encoded_feature_names, index=combined_df.index)
    combined_df = pd.concat([combined_df.drop(columns=['coin', 'symbol']), encoded_df], axis=1)

print("\nDataFrame head after feature scaling and final encoding:")
print(combined_df.head().to_markdown(index=False, numalign="left", stralign="left"))
print(f"\nFinal preprocessed DataFrame shape: {combined_df.shape}")


# --- 6. Model Selection and Data Splitting ---
print("\n--- Step 6: Model Selection and Data Splitting ---")
# RandomForestRegressor is chosen for its robustness and performance in regression tasks.

# Prepare features (X) and target (y)
# Drop the 'date' column as it's not a direct feature in its original form,
# and '24h_volume' is our target variable.
# 'time_since_epoch' is retained as a numerical feature.
X = combined_df.drop(columns=['date', '24h_volume'])
y = combined_df['24h_volume']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


# --- 7. Model Training (Initial) ---
print("\n--- Step 7: Model Training (Initial) ---")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Initial RandomForestRegressor model trained.")


# --- 8. Model Evaluation (Initial) ---
print("\n--- Step 8: Model Evaluation (Initial) ---")
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Initial Model Evaluation Metrics:")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R²): {r2:.4f}")

# Display top feature importances from the initial model
print("\nTop 10 Feature Importances (Initial Model):")
feature_importances = pd.Series(model.feature_importances_, index=X.columns)
print(feature_importances.nlargest(10).to_markdown(numalign="left", stralign="left"))


# --- 9. Hyperparameter Tuning ---
print("\n--- Step 9: Hyperparameter Tuning (RandomizedSearchCV) ---")
# Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': randint(50, 200),
    'max_features': ['sqrt', 'log2', None],
    'max_depth': randint(10, 50),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5)
}

# Initialize RandomizedSearchCV
# Setting n_jobs=1 to avoid multiprocessing/pickling issues in this environment
random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=10,  # Number of random combinations to try
    cv=3,       # 3-fold cross-validation
    verbose=2,  # Set to 0 for less verbose output
    random_state=42,
    n_jobs=1,   # Use 1 core to prevent potential multiprocessing issues
    scoring='r2' # Optimize for R-squared
)

print("Starting hyperparameter tuning...")
random_search.fit(X_train, y_train)
print("Hyperparameter tuning completed.")

# Get the best estimator from RandomizedSearchCV
best_model = random_search.best_estimator_

print("\nBest Hyperparameters found:")
print(random_search.best_params_)

# Evaluate the best model
y_pred_tuned = best_model.predict(X_test)

mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
mse_tuned = mean_squared_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mse_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print("\nTuned Model Evaluation Metrics (RandomForestRegressor):")
print(f"Mean Absolute Error (MAE): {mae_tuned:.4f}")
print(f"Mean Squared Error (MSE): {mse_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_tuned:.4f}")
print(f"R-squared (R²): {r2_tuned:.4f}")


# --- 10. Model Testing & Validation (Visualizations) ---
print("\n--- Step 10: Model Testing & Validation (Visualizations) ---")
# (Note: For actual plots, uncomment and run in an environment with matplotlib/seaborn)
# import matplotlib.pyplot as plt
# import seaborn as sns

# print("\nGenerating Actual vs. Predicted plot and Residual plots...")
# # Plotting Actual vs. Predicted values
# plt.figure(figsize=(12, 7))
# sns.scatterplot(x=y_test, y=y_pred_tuned, alpha=0.6)
# plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Diagonal line
# plt.title('Actual vs. Predicted 24h Volume (Tuned Model)')
# plt.xlabel('Actual 24h Volume (Scaled)')
# plt.ylabel('Predicted 24h Volume (Scaled)')
# plt.grid(True)
# plt.show()
#
# # Plotting Residuals Distribution
# residuals = y_test - y_pred_tuned
# plt.figure(figsize=(12, 7))
# sns.histplot(residuals, kde=True, bins=50)
# plt.title('Distribution of Residuals')
# plt.xlabel('Residuals (Actual - Predicted)')
# plt.ylabel('Frequency')
# plt.grid(True)
# plt.show()
#
# # Plotting Residuals vs. Predicted values
# plt.figure(figsize=(12, 7))
# sns.scatterplot(x=y_pred_tuned, y=residuals, alpha=0.6)
# plt.axhline(y=0, color='r', linestyle='--', lw=2)
# plt.title('Residuals vs. Predicted Values')
# plt.xlabel('Predicted 24h Volume (Scaled)')
# plt.ylabel('Residuals')
# plt.grid(True)
# plt.show()

# Display some actual vs. predicted values for a quick check
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_tuned})
print("\nSome Actual vs. Predicted 24h Volume values (first 10):")
print(results_df.head(10).to_markdown(numalign="left", stralign="left"))


# --- 11. Model Persistence (Saving the Model for Deployment) ---
print("\n--- Step 11: Model Persistence (Saving for Deployment) ---")
# Save the trained model and the list of feature columns
joblib.dump(best_model, 'liquidity_model.pkl')
joblib.dump(X.columns.tolist(), 'model_features.pkl')

print("Tuned model and feature names saved successfully:")
print("- 'liquidity_model.pkl' (the trained RandomForestRegressor model)")
print("- 'model_features.pkl' (list of feature column names for consistent input)")

print("\n--- Project Execution Complete ---")
print("The ML project for Cryptocurrency Liquidity Prediction has been fully executed.")
print("The trained model 'liquidity_model.pkl' is now ready for conceptual deployment.")


--- Step 1: Data Collection & Initial Setup ---
CSV files loaded successfully.

Combined DataFrame head (before extensive preprocessing):
| coin     | symbol   | price    | 1h     | 24h    | 7d    | 24h_volume   | mkt_cap     | date                |
|:---------|:---------|:---------|:-------|:-------|:------|:-------------|:------------|:--------------------|
| Bitcoin  | BTC      | 40859.5  | 0.022  | 0.03   | 0.055 | 3.53908e+10  | 7.70991e+11 | 2022-03-16 00:00:00 |
| Ethereum | ETH      | 2744.41  | 0.024  | 0.034  | 0.065 | 1.97487e+10  | 3.27104e+11 | 2022-03-16 00:00:00 |
| Tether   | USDT     | 1        | -0.001 | -0.001 | 0     | 5.7935e+10   | 7.99652e+10 | 2022-03-16 00:00:00 |
| BNB      | BNB      | 383.43   | 0.018  | 0.028  | 0.004 | 1.39585e+09  | 6.40438e+10 | 2022-03-16 00:00:00 |
| USD Coin | USDC     | 0.999874 | -0.001 | 0      | -0    | 3.87227e+09  | 5.22221e+10 | 2022-03-16 00:00:00 |

Initial combined DataFrame shape: (1000, 9)

Initial missing values overview: